# Video Highlight Pipeline: Long Video → Douyin-style Short

Turn long videos (up to 60 min) into 2-5 minute highlight reels with AI narration.

**Features:**
- Qwen2.5-VL video analysis (temporal grounding)
- LLM-generated narration script with emotion vectors
- IndexTTS2 voice cloning with emotion injection
- Portrait mode, transitions, color grading, ASS subtitles
- Auto-resume from checkpoint if interrupted

**Prerequisites:**
1. Set `LLM_API_KEY` in [Colab Secrets](https://colab.research.google.com/notebooks/secrets.ipynb)
2. Upload a **reference audio** file for TTS voice cloning

## 0. Mount Google Drive & Set Model Cache

All large model files (IndexTTS2, Whisper, Qwen2.5-VL) are cached on Google Drive so they persist across sessions.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# --- Model cache on Google Drive (avoids re-downloading every session) ---
DRIVE_CACHE = "/content/drive/MyDrive/models_cache"
os.makedirs(f"{DRIVE_CACHE}/hf_home", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/torch_home", exist_ok=True)

# HuggingFace models: Whisper, Qwen2.5-VL, IndexTTS2
os.environ["HF_HOME"] = f"{DRIVE_CACHE}/hf_home"
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch_home"

print(f"Model cache: {DRIVE_CACHE}")
print(f"HF_HOME:    {os.environ['HF_HOME']}")
print(f"TORCH_HOME: {os.environ['TORCH_HOME']}")

## 1. Install dependencies

In [ ]:
!nvidia-smi
import sys
print(f"Python {sys.version}")

In [ ]:
# Clone the repo
%cd /content
!git clone -b py3.12 https://github.com/deluxebear/index-tts.git 2>/dev/null || (cd /content/index-tts && git pull)
%cd /content/index-tts

In [ ]:
%cd /content/index-tts

# Uninstall conflicting Colab packages + install project deps
!pip uninstall -y torch torchvision torchaudio tensorflow keras tensorboard protobuf 2>/dev/null
!pip install ninja
!pip install -e ".[webui]" --extra-index-url https://download.pytorch.org/whl/cu128

# Highlight pipeline deps (torchvision required by Qwen2.5-VL processor)
!pip install -q whisperx soundfile openai
!pip install -q transformers>=4.52.1 qwen-vl-utils accelerate
!pip install -q torchvision --extra-index-url https://download.pytorch.org/whl/cu128

# Fix: whisperx may pull in numpy 2.x but numba 0.60 needs numpy < 2.0
!pip install "numpy<2.0"

# Restart runtime so new packages take effect
print("Restarting runtime to apply package changes...")
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

### After runtime restart, run this cell to restore environment

The install cell above restarts the runtime. Run this cell to re-mount Drive and verify packages.

In [ ]:
import os
from google.colab import drive

# Re-mount Drive and restore env vars after runtime restart
drive.mount('/content/drive')

DRIVE_CACHE = "/content/drive/MyDrive/models_cache"
os.environ["HF_HOME"] = f"{DRIVE_CACHE}/hf_home"
os.environ["TORCH_HOME"] = f"{DRIVE_CACHE}/torch_home"

%cd /content/index-tts

# Verify packages loaded correctly
import torch, numpy, numba
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()}")
print(f"numpy={numpy.__version__}")
print(f"numba={numba.__version__}")
print("All good!")

## 2. Download IndexTTS2 checkpoints

Checkpoints are downloaded to Google Drive and symlinked to `checkpoints/`.

In [ ]:
import os

DRIVE_CKPT = f"{DRIVE_CACHE}/indextts2_checkpoints"
LOCAL_CKPT = "/content/index-tts/checkpoints"

# Download to Drive (persists across sessions)
if not os.path.exists(f"{DRIVE_CKPT}/config.yaml"):
    print("Downloading IndexTTS2 checkpoints to Google Drive (first time only)...")
    !huggingface-cli download IndexTeam/IndexTTS-2 --local-dir "{DRIVE_CKPT}"
else:
    print(f"Checkpoints already cached at {DRIVE_CKPT}")

# Symlink so the pipeline finds them at the expected local path
if os.path.islink(LOCAL_CKPT):
    os.unlink(LOCAL_CKPT)
elif os.path.exists(LOCAL_CKPT):
    import shutil
    shutil.rmtree(LOCAL_CKPT) if os.path.isdir(LOCAL_CKPT) else os.remove(LOCAL_CKPT)
os.symlink(DRIVE_CKPT, LOCAL_CKPT)
print(f"Symlinked: {LOCAL_CKPT} -> {DRIVE_CKPT}")

# Verify
!ls -la checkpoints/config.yaml

## 3. Configuration

In [ ]:
from google.colab import userdata

LLM_API_KEY = userdata.get('LLM_API_KEY')  # LLM API key — for script generation

# --- LLM provider (uncomment one) ---
LLM_API_BASE = "https://api.openai.com/v1"       ; LLM_MODEL = "gpt-4o-mini"
# LLM_API_BASE = "https://api.deepseek.com/v1"    ; LLM_MODEL = "deepseek-chat"
# LLM_API_BASE = "https://generativelanguage.googleapis.com/v1beta/openai/" ; LLM_MODEL = "gemini-2.0-flash"

# --- Highlight settings ---
VL_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"  # Video understanding model (~18GB VRAM)
TARGET_DURATION = 180   # Target output duration in seconds (2-5 min recommended)
USE_FP16 = True         # FP16 inference (faster, less VRAM)
WORK_DIR = "highlight_workspace"
CLEANUP = False         # Delete intermediate files after completion

print(f"LLM: {LLM_MODEL} @ {LLM_API_BASE}")
print(f"VL Model: {VL_MODEL}")
print(f"Target duration: {TARGET_DURATION}s ({TARGET_DURATION/60:.1f}min)")
print(f"FP16: {USE_FP16}, Cleanup: {CLEANUP}")

## 4. Pre-download heavy models (optional)

Run this cell once to download Whisper and Qwen2.5-VL models to Google Drive. Subsequent sessions will reuse cached models.

In [ ]:
# Pre-download Whisper model (~3GB)
import whisperx
print("Loading Whisper model (cached on Drive)...")
_model = whisperx.load_model("large-v2", "cuda", compute_type="float16")
del _model
print("Whisper model cached.")

# Pre-download Qwen2.5-VL model (~18GB)
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
print(f"Loading {VL_MODEL} (cached on Drive)...")
_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VL_MODEL, torch_dtype="auto", device_map="auto"
)
_processor = AutoProcessor.from_pretrained(VL_MODEL)
del _model, _processor
print("Qwen2.5-VL model cached.")

import torch; torch.cuda.empty_cache()
import gc; gc.collect()
print("\nAll models cached on Google Drive. Future sessions will start faster.")

## 5. Upload video & reference audio, then run

In [ ]:
from highlight_pipeline import highlight_video
from google.colab import files
from pathlib import Path

# === Upload video ===
print("Upload your video file:")
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# === Upload reference audio (for TTS voice cloning) ===
print("\nUpload reference audio (5-15s WAV of target voice):")
uploaded_ref = files.upload()
ref_audio = list(uploaded_ref.keys())[0]

# === Option: Use Google Drive paths instead (uncomment below) ===
# video_path = "/content/drive/MyDrive/videos/ted_talk.mp4"
# ref_audio = "/content/drive/MyDrive/voices/narrator.wav"

# === Run (auto-resumes if interrupted) ===
stem = Path(video_path).stem
output_path = f"/content/{stem}_highlight.mp4"

highlight_video(
    video_path=video_path,
    ref_audio=ref_audio,
    output_path=output_path,
    work_dir=WORK_DIR,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    vl_model=VL_MODEL,
    model_dir="checkpoints",
    target_duration=TARGET_DURATION,
    use_fp16=USE_FP16,
    cleanup=CLEANUP,
)

# Play result inline
from IPython.display import Video, display
display(Video(output_path, embed=True, width=640))

In [ ]:
# Download the highlight video
files.download(output_path)

## 6. Batch mode (Google Drive)

Process all videos in a directory. Output goes to a parallel folder.

In [ ]:
from highlight_pipeline import highlight_batch

INPUT_DIR  = "/content/drive/MyDrive/videos/input"   # <- your input folder
OUTPUT_DIR = "/content/drive/MyDrive/videos/highlights"  # <- highlight output folder
REF_AUDIO  = "/content/drive/MyDrive/voices/narrator.wav"  # <- reference voice

highlight_batch(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    ref_audio=REF_AUDIO,
    work_dir=WORK_DIR,
    llm_api_key=LLM_API_KEY,
    llm_api_base=LLM_API_BASE,
    llm_model=LLM_MODEL,
    vl_model=VL_MODEL,
    model_dir="checkpoints",
    target_duration=TARGET_DURATION,
    use_fp16=USE_FP16,
    cleanup=CLEANUP,
)
print(f"\nAll done! Check: {OUTPUT_DIR}")

---
## 7. Inspect intermediate results

The pipeline saves intermediate files for debugging. Check the `highlight_workspace/` directory.

In [ ]:
import json, os
from IPython.display import Audio, display

# Find the most recent work directory
work_dirs = sorted(
    [d for d in os.listdir(WORK_DIR) if os.path.isdir(f"{WORK_DIR}/{d}")]
) if os.path.isdir(WORK_DIR) else []

if work_dirs:
    work_dir = f"{WORK_DIR}/{work_dirs[-1]}"
    print(f"Work directory: {work_dir}")
    print(f"Contents: {os.listdir(work_dir)}")

    # Show transcript
    transcript_path = f"{work_dir}/transcript.json"
    if os.path.exists(transcript_path):
        with open(transcript_path) as f:
            segments = json.load(f)
        print(f"\n--- Transcript ({len(segments)} segments) ---")
        for s in segments[:10]:
            print(f"  {s['start']:.1f}-{s['end']:.1f}s: {s['text']}")
        if len(segments) > 10:
            print(f"  ... and {len(segments)-10} more")

    # Show script
    script_path = f"{work_dir}/script.json"
    if os.path.exists(script_path):
        with open(script_path) as f:
            script = json.load(f)
        print(f"\n--- Script ({len(script)} segments) ---")
        for i, seg in enumerate(script[:10]):
            narration = seg.get('narration', '')
            start = seg.get('start', 0)
            end = seg.get('end', 0)
            emotion = seg.get('emotion', {})
            print(f"  #{i} [{start:.1f}-{end:.1f}s] {narration[:60]}")
            if emotion:
                top = sorted(emotion.items(), key=lambda x: -x[1])[:3]
                print(f"       emotion: {', '.join(f'{k}={v}' for k,v in top)}")

    # Show VL analysis
    analysis_path = f"{work_dir}/vl_analysis.json"
    if os.path.exists(analysis_path):
        with open(analysis_path) as f:
            analysis = json.load(f)
        events = analysis.get('events', [])
        print(f"\n--- VL Analysis ({len(events)} events) ---")
        for e in events[:10]:
            print(f"  [{e.get('start',0):.1f}-{e.get('end',0):.1f}s] "
                  f"imp={e.get('importance',0)} {e.get('description','')[:50]}")

    # Play narration audio files
    tts_dir = f"{work_dir}/tts_output"
    if os.path.isdir(tts_dir):
        tts_files = sorted(f for f in os.listdir(tts_dir) if f.endswith('.wav'))[:3]
        if tts_files:
            print(f"\n--- Narration samples ({len(tts_files)} shown) ---")
            for f in tts_files:
                print(f"  {f}")
                display(Audio(f"{tts_dir}/{f}"))
else:
    print("No work directories found. Run the pipeline first.")